# Colab T4: RMT / GRT Copy 초기 수렴 비교
런타임을 T4 GPU로 선택하고, 최신 수정본을 `/content/GRT`에 준비하세요. 기존 `smoke_test.ipynb`의 CPU 테스트가 통과한 뒤 실행합니다.
표준 small(4층, D=256, FF=1024), N=128, M=32, V=1024, Copy T=4를 유지합니다. 두 모델에 FP32, effective batch 8, 5,000 optimizer update, 동일 seed/sample stream을 적용합니다. validation/test는 각각 256개입니다.
FP32로 수렴을 먼저 확인합니다. 5,000 update 안에 CE≤0.05가 달성된다고 가정하지 않습니다. single-seed 초기 탐색이며 표준 50,000-step 실험과 구분합니다. Colab 세션 내 완료 시간은 보장하지 않습니다.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import os, sys, subprocess, json, importlib
REPO_DIR = Path('/content/GRT')
USE_DRIVE = True  # False이면 결과가 /content에만 저장됩니다.
assert (REPO_DIR / 'configs/colab_t4_copy.yaml').exists(), '최신 수정본과 REPO_DIR를 확인하세요.'
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)
# Editable install의 .pth는 이미 실행 중인 커널에서 자동으로 다시 로드되지 않을 수 있습니다.
SOURCE_DIR = (REPO_DIR / 'src').resolve()
assert (SOURCE_DIR / 'grt/__init__.py').is_file(), '저장소의 src/grt 경로를 확인하세요.'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
importlib.invalidate_caches()
import torch
assert torch.cuda.is_available(), 'Colab 런타임을 T4 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/GRT/runs/t4_copy_5000_fp32_seed1234')
else:
    RUN_ROOT = REPO_DIR / 'runs/t4_copy_5000_fp32_seed1234'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('실험 저장 경로:', RUN_ROOT)
from grt.config import load_config, validate_config
from grt.checkpoint import load_checkpoint
def train_or_resume(model_name):
    assert model_name in {'rmt', 'grt'}
    run_dir = RUN_ROOT / model_name
    configs = ['configs/base.yaml', f'configs/{model_name}.yaml', 'configs/copy.yaml', 'configs/colab_t4_copy.yaml']
    expected = asdict(validate_config(load_config(configs, output_dir=run_dir)))
    candidates = {run_dir / 'last.pt', run_dir / 'best.pt'}
    periodic = sorted(run_dir.glob('step_*.pt'))
    if periodic:
        candidates.add(periodic[-1])
    saved = []
    for path in candidates:
        if not path.exists():
            continue
        state = load_checkpoint(path)
        recorded = state['config']
        recorded['run']['output_dir'] = str(run_dir)
        assert recorded == expected, f'{path}: 설정이 다릅니다. 새 RUN_ROOT를 사용하세요.'
        saved.append((state['global_step'], str(path)))
        del state
    if saved:
        step, checkpoint = max(saved)
        if step >= expected['training']['max_steps'] and (run_dir / 'evaluation.json').exists():
            print(model_name, '이미 완료됨:', run_dir)
            return
        print(model_name, '재개:', checkpoint, 'step:', step, flush=True)
        arguments = ['scripts/train.py', '--resume', checkpoint, '--device', 'cuda']
    else:
        print(model_name, '새 학습:', run_dir, flush=True)
        arguments = ['scripts/train.py', '--config', *configs, '--output-dir', str(run_dir), '--device', 'cuda']
    subprocess.run([sys.executable, '-u', *arguments], cwd=REPO_DIR, check=True)


RMT baseline을 먼저 실행하고 아래 GRT 셀을 실행합니다. 각 셀은 5,000 update 전체 예산을 사용하고 종료 시 best checkpoint의 T=4/10/20 성능을 평가합니다. validation 지표는 매 250 update 콘솔에 출력됩니다.
중단 후 같은 저장 경로로 setup 셀과 해당 모델 셀을 다시 실행하면 저장된 가장 최신 checkpoint에서 재개합니다. 설정을 바꿀 때는 두 모델 모두 새 RUN_ROOT로 시작하세요. runtime이 종료되어 로컬 코드가 사라졌다면 저장소를 다시 준비해야 합니다.

In [ ]:
train_or_resume('rmt')


In [ ]:
train_or_resume('grt')


In [ ]:
# 이 셀은 한 모델이 중단된 뒤에도 별도로 실행할 수 있습니다.
import matplotlib.pyplot as plt
summary = {'stage': 'initial_convergence_pilot', 'task': 'copy', 'seed': 1234,
           'precision': 'fp32', 'budget': 5000, 'effective_batch': 8, 'models': {}}
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for model_name in ['rmt', 'grt']:
    run_dir = RUN_ROOT / model_name
    path = run_dir / 'metrics.jsonl'
    if not path.exists():
        print(model_name, '아직 로그 없음')
        continue
    records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    by_step = {r['global_step']: r for r in records if 'val/loss' in r}
    validation = [by_step[s] for s in sorted(by_step)]
    if not validation:
        print(model_name, '아직 validation 없음')
        continue
    steps = [r['global_step'] for r in validation]
    losses = [r['val/loss'] for r in validation]
    reached = [r for r in validation if r['val/loss'] <= 0.05]
    completed = max(r['global_step'] for r in records)
    row = {'completed_updates': completed, 'best_validation_ce': min(losses),
           'last_validation_ce': losses[-1],
           'last_token_accuracy': validation[-1]['val/token_accuracy'],
           'last_exact_match': validation[-1]['val/exact_match'],
           'first_target_step': reached[0]['global_step'] if reached else None,
           'status': 'target_reached' if reached else ('not_reached_within_budget' if completed >= 5000 else 'incomplete_run')}
    evaluation = run_dir / 'evaluation.json'
    if evaluation.exists():
        result = json.loads(evaluation.read_text())
        row['checkpoint_step'] = result['checkpoint_step']
        row['evaluation'] = result['results']
        print(model_name, [{k: r[k] for k in ['T', 'loss', 'token_accuracy', 'exact_match']} for r in result['results']])
    summary['models'][model_name] = row
    axes[0].plot(steps, [max(x, 1e-8) for x in losses], label=model_name)
    axes[1].plot(steps, [r['val/token_accuracy'] for r in validation], label=model_name)
    axes[2].plot(steps, [r['val/exact_match'] for r in validation], label=model_name)
axes[0].set_yscale('log')
axes[0].axhline(0.05, color='gray', linestyle='--', label='CE target 0.05')
for ax, title in zip(axes, ['Validation masked CE', 'Validation token accuracy', 'Validation exact match']):
    ax.set(title=title, xlabel='Optimizer updates')
    ax.legend()
    ax.grid(alpha=0.2)
for ax in axes[1:]:
    ax.set_ylim(0, 1.02)
fig.savefig(RUN_ROOT / 'convergence.png', dpi=150)
plt.show()
(RUN_ROOT / 'comparison.json').write_text(json.dumps(summary, indent=2, allow_nan=False), encoding='utf-8')
print(json.dumps({m: {k: v for k, v in row.items() if k != 'evaluation'} for m, row in summary['models'].items()}, indent=2))
print('공유할 결과:', RUN_ROOT / 'comparison.json', RUN_ROOT / 'convergence.png')
